In [2]:
import pandas as pd

import numpy as np

在数据分析中，经常会遇到这样的情况：

    经常需要将数据根据某个（多个）字段划分为不同的群体（group）进行分析
    
    比如，电商领域将全国的总销售额根据省份进行划分，分析各省销售额的变化情况；
         社交领域将用户根据画像（性别、年龄）进行细分，研究用户的使用情况和偏好等
    
    在 Pandas 中，要完成数据的分组操作，需要使用 groupby() 函数，它和 SQL 的GROUP BY操作非常相似。

在划分出来的组（group）上应用一些统计函数，从而达到数据分析的目的，比如对分组数据进行聚合、转换，或者过滤。这个过程主要包含以下三步：

SAC过程：
- **拆分**（Spliting）：表示对数据进行分组；
- **应用**（Applying）：对分组数据应用聚合函数，进行相应计算；
- **合并**（Combining）：最后汇总计算结果。

apply过程：

- **整合**（Aggregation）——即分组计算统计量（如求均值、求每组元素个数）
- **变换**（Transformation）——即分组对每个单元的数据进行操作（如元素标准化）
- **过滤**（Filtration）——即按照某些规则筛选出一些组（如选出组内某一指标小于50的组）
- **综合问题**——即前面提及的三种问题的混合

模拟生成的10个样本数据，代码和数据如下：

In [9]:
company = ["A", "B", "C"]
data = pd.DataFrame({
    "company": [company[x] for x in np.random.randint(0, len(company), 10)],
    "salary": np.random.randint(5, 50, 10),
    "age": np.random.randint(15, 50, 10)})

data

,company,salary,age
0,A,45,28
1,A,35,21
2,B,45,36
3,A,44,19
4,B,48,46
5,C,25,23
6,A,20,48
7,C,44,21
8,B,32,32
9,C,46,33


# Groupby的基本原理

在pandas中，实现分组操作的代码很简单，仅需一行代码，在这里，将上面的数据集按照company字段进行划分：

In [10]:
group = data.groupby("company")
group

> 将上述代码输入ipython后，会得到一个DataFrameGroupBy对象，该对象本身不会返回任何东西，只有当相应的方法被调用才会起作用。

ipython所返回的结果是其内存地址，并不利于直观地理解，为了看看group内部究竟是什么，这里把group转换成list的形式来看一看：

In [11]:
list(group)

[('A',
    company  salary  age
  0       A      45   28
  1       A      35   21
  3       A      44   19
  6       A      20   48),
 ('B',
    company  salary  age
  2       B      45   36
  4       B      48   46
  8       B      32   32),
 ('C',
    company  salary  age
  5       C      25   23
  7       C      44   21
  9       C      46   33)]

In [12]:
group.groups

{'A': Int64Index([0, 1, 3, 6], dtype='int64'),
 'B': Int64Index([2, 4, 8], dtype='int64'),
 'C': Int64Index([5, 7, 9], dtype='int64')}

转换成列表的形式后，可以看到，列表由三个元组组成，每个元组中，

第一个元素是组别（这里是按照company进行分组，所以最后分为了A,B,C），

第二个元素的是对应组别下的DataFrame，整个过程可以图解如下：

<img src="images/20220424234526.png" style="width:60%"/>

总结来说，groupby的过程就是将原有的DataFrame按照groupby的字段（这里是company），划分为若干个分组DataFrame，被分为多少个组就有多少个分组DataFrame。所以说，在groupby之后的一系列操作（如agg、apply等），均是基于子DataFrame的操作。

理解了这点，也就基本摸清了Pandas中groupby操作的主要原理。

## 分组函数的基本内容

### 获取某一个组

In [14]:
group.get_group('A')

,company,salary,age
0,A,45,28
1,A,35,21
3,A,44,19
6,A,20,48


### 组容量与组数

In [15]:
group.size()

company
A    4
B    3
C    3
dtype: int64

In [16]:
group.ngroups

3

### 组的遍历

In [18]:
for n, g in group:
    print(n)
    display(g)

A


,company,salary,age
0,A,45,28
1,A,35,21
3,A,44,19
6,A,20,48


B


,company,salary,age
2,B,45,36
4,B,48,46
8,B,32,32


C


,company,salary,age
5,C,25,23
7,C,44,21
9,C,46,33


下面来讲讲groupby之后的常见操作。

# agg 聚合操作

## 常用聚合函数

聚合(Aggregation)操作是groupby后非常常见的操作，会写SQL的朋友对此应该是非常熟悉了。聚合操作可以用来求和、均值、最大值、最小值等，下面的表格列出了Pandas中常见的聚合操作。

<img src="images/20220424234734.png" style="height:60%" />

针对样例数据集，<b>如果我想求不同公司员工的平均年龄和平均薪水</b>，可以按照下方的代码进行：

In [19]:
data.groupby("company").agg('mean')

,salary,age
company,,
A,36.000000,29.000000
B,41.666667,38.000000
C,38.333333,25.666667


In [20]:
data.groupby("company").mean()

,salary,age
company,,
A,36.000000,29.000000
B,41.666667,38.000000
C,38.333333,25.666667


**同时使用多个聚合函数**

In [21]:
data.groupby('company').agg(['mean', 'sum', 'max'])

salary                 age         
              mean  sum max       mean  sum max
company                                        
A        36.000000  144  45  29.000000  116  48
B        41.666667  125  48  38.000000  114  46
C        38.333333  115  46  25.666667   77  33

如果想对针对**不同的列求不同的值**，比如要计算不同公司员工的平均年龄以及薪水的中位数和平均值，可以利用字典进行聚合操作的指定：

In [23]:
data.groupby('company').agg({'salary':['median', 'mean'], 'age':'mean'})

salary                   age
        median       mean       mean
company                             
A         39.5  36.000000  29.000000
B         45.0  41.666667  38.000000
C         44.0  38.333333  25.666667

**使用自定义函数**

通过agg可以容易实现组内极差的计算

In [31]:
data.groupby('company').agg([('range', lambda x: x.max()-x.min()), 'max', 'min'])

salary           age        
         range max min range max min
company                             
A           25  45  20    29  48  19
B           16  48  32    14  46  32
C           21  46  25    12  33  21

agg聚合过程可以图解如下：

<img src="images/20220424235424.png" style="width:60%"/>

# transform 转换值

transform是一种什么数据操作？

和agg有什么区别呢？


为了更好地理解transform和agg的不同，下面从实际的应用场景出发进行对比。

在上面的agg中，我们学会了如何求不同公司员工的平均薪水，

<b>如果现在需要在原数据集中新增一列avg_salary，代表员工所在的公司的平均薪水（相同公司的员工具有一样的平均薪水），该怎么实现呢？</b>

如果按照正常的步骤来计算，需要先求得不同公司的平均薪水，然后按照员工和公司的对应关系填充到对应的位置，不用transform的话

In [38]:
data.groupby('company')['salary'].mean()

company
A    36.000000
B    41.666667
C    38.333333
Name: salary, dtype: float64

In [40]:
# to_dict将表格中的数据转换成字典格式
avg_salary_dict = data.groupby('company')['salary'].mean().to_dict()
avg_salary_dict

{'A': 36.0, 'B': 41.666666666666664, 'C': 38.333333333333336}

In [41]:
# map()函数可以用于Series对象或DataFrame对象的一列，
# 接收函数作为或字典对象作为参数，返回经过函数或字典映射处理后的值。
data['avg_salary'] = data['company'].map(avg_salary_dict)
data

,company,salary,age,avg_salary
0,A,45,28,36.000000
1,A,35,21,36.000000
2,B,45,36,41.666667
3,A,44,19,36.000000
4,B,48,46,41.666667
5,C,25,23,38.333333
6,A,20,48,36.000000
7,C,44,21,38.333333
8,B,32,32,41.666667
9,C,46,33,38.333333


如果使用transform的话，仅需要一行代码：

In [42]:
data.groupby('company')['salary'].transform('mean')

0    36.000000
1    36.000000
2    41.666667
3    36.000000
4    41.666667
5    38.333333
6    36.000000
7    38.333333
8    41.666667
9    38.333333
Name: salary, dtype: float64

In [43]:
data['avg_salary1'] = data.groupby('company')['salary'].transform('mean')
data
# 分组之后的数据3条  A--值  B--值  C--对应的值

,company,salary,age,avg_salary,avg_salary1
0,A,45,28,36.000000,36.000000
1,A,35,21,36.000000,36.000000
2,B,45,36,41.666667,41.666667
3,A,44,19,36.000000,36.000000
4,B,48,46,41.666667,41.666667
5,C,25,23,38.333333,38.333333
6,A,20,48,36.000000,36.000000
7,C,44,21,38.333333,38.333333
8,B,32,32,41.666667,41.666667
9,C,46,33,38.333333,38.333333


还是以图解的方式来看看进行groupby后transform的实现过程（为了更直观展示，图中加入了company列，实际按照上面的代码只有salary列）：

<img src="images/20220425000807.png" style="width:80%"/>

图中的大方框是transform和agg所不一样的地方，对agg而言，会计算得到A，B，C公司对应的均值并直接返回，

但对transform而言，则会对每一条数据求得相应的结果，同一组内的样本会有相同的值，

组内求完均值后会`按照原索引的顺序`返回结果，如果有不理解的可以拿这张图和agg那张对比一下。

# 过滤

filter函数是用来筛选某些组的（务必记住结果是组的全体），因此传入的值应当是布尔标量

In [79]:
data.groupby('company')[['company', 'salary', 'age']].filter(lambda x:(x['age']>20).all())

,company,salary,age
2,B,45,36
4,B,48,46
5,C,25,23
7,C,44,21
8,B,32,32
9,C,46,33


# apply 
它相比agg和transform而言更加灵活，能够传入任意自定义的函数，实现复杂的数据操作

对于groupby后的apply，以分组后的子DataFrame作为参数传入指定函数的，基本操作单位是DataFrame

In [81]:
list(data.groupby('company',as_index=False))

[('A',
    company  salary  age  avg_salary  avg_salary1
  0       A      45   28        36.0         36.0
  1       A      35   21        36.0         36.0
  3       A      44   19        36.0         36.0
  6       A      20   48        36.0         36.0),
 ('B',
    company  salary  age  avg_salary  avg_salary1
  2       B      45   36   41.666667    41.666667
  4       B      48   46   41.666667    41.666667
  8       B      32   32   41.666667    41.666667),
 ('C',
    company  salary  age  avg_salary  avg_salary1
  5       C      25   23   38.333333    38.333333
  7       C      44   21   38.333333    38.333333
  9       C      46   33   38.333333    38.333333)]

<b>假设我现在需要获取各个公司年龄最大的员工的数据，该怎么实现呢？</b>

In [85]:
def get_oldest_staff(x):
    # 输入的数据按照age字段进行排序
    df = x.sort_values(by='age')
    # 返回最后一条数据
    return df.iloc[-1]

oldest_staff = data.groupby('company', as_index=False).apply(get_oldest_staff)
oldest_staff

,company,salary,age,avg_salary,avg_salary1
0,A,20,48,36.000000,36.000000
1,B,48,46,41.666667,41.666667
2,C,46,33,38.333333,38.333333


这样便得到了每个公司年龄最大的员工的数据，整个流程图解如下：
<img src="images/20220425002232.png" style="width:80%"/>

> 虽然说apply拥有更大的灵活性，但apply的运行效率会比agg和transform更慢。所以，groupby之后能用agg和transform解决的问题还是优先使用这两个方法，实在解决不了了才考虑使用apply进行操作